# v5 Cell Comparison (Post-Eval)

Loads a `RunReport` from `cell_runner.run()` and visualizes:
- Per-cell McNemar test results (b, c, χ², p_corrected)
- Cohen's h effect size per cell with 95% bootstrap CI
- Aggregate result

**Templated; not runnable until Phase D evaluation produces a real RunReport.**

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Update path once Phase D evaluation produces a report
REPORT_PATH = Path('../RESULTS/v5_phase1_run_report.json')
if not REPORT_PATH.exists():
    print(f'No report at {REPORT_PATH} — run Phase D evaluation first')
else:
    with open(REPORT_PATH) as f:
        report = json.load(f)
    print(report['aggregate'])

In [ ]:
# Per-cell McNemar table
rows = []
for c in report['cells']:
    m = c.get('mcnemar')
    if m is None:
        continue
    rows.append({
        'cell': c['cell'],
        'n': m['n_chains'],
        'b': m['n_discordant_base_only'],
        'c': m['n_discordant_intervention_only'],
        'chi2': m['statistic'],
        'p_raw': m['p_value'],
        'p_corr': m['p_value_corrected'],
        'h': m['effect_size_h'],
        'ci_lo': m['ci_lower'],
        'ci_hi': m['ci_upper'],
        'sig': m['significant'],
    })
df = pd.DataFrame(rows)
df

In [ ]:
# Per-cell effect size with bootstrap CI
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(df))
ax.errorbar(x, df['h'], yerr=[df['h'] - df['ci_lo'], df['ci_hi'] - df['h']],
             fmt='o', capsize=8, color=['green' if s else 'gray' for s in df['sig']])
ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(df['cell'])
ax.set_ylabel("Cohen's h")
ax.set_title('Per-cell effect size with 95% bootstrap CI')
plt.tight_layout()
plt.show()